In [5]:
import pandas as pd

In [6]:
#1数据导入
titanic = pd.read_csv('tianchi_train.csv')
titanic

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [7]:
x = titanic[['Pclass','Age','Sex']]
y = titanic['Survived']

In [8]:
#2数据处理
x['Age'].fillna(x['Age'].mean(),inplace=True)

C:\Users\86157\AppData\Local\Temp\ipykernel_19552\1058473533.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  x['Age'].fillna(x['Age'].mean(),inplace=True)
C:\Users\86157\AppData\Local\Temp\ipykernel_19552\1058473533.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x['Age'].fillna(x['Age'].mean(),inplace=True)


In [9]:
#转行成字典
x = x.to_dict(orient='records')

In [10]:
#划分数据集
from sklearn.model_selection import train_test_split

In [11]:
x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=22)

In [12]:
#字典特征抽取
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeClassifier,export_graphviz

In [13]:
transfer = DictVectorizer()
x_train = transfer.fit_transform(x_train)
x_test= transfer.transform(x_test)

In [14]:

#3决策树预估器
estimator = DecisionTreeClassifier(criterion='entropy')
estimator.fit(x_train,y_train)
#4模型评估

# 模型评估
# 方法一：直接比对真实值和预测值
y_predict = estimator.predict(x_test)
print('y_predict: \n', y_predict)
print('直接比对真实值和预测值: \n', y_test == y_predict)
# 方法二:计算准确率
score = estimator.score(x_test, y_test)
print('准确率: \n', score)
    
#可视化决策树
export_graphviz(estimator,out_file='titanic_tree.dot',feature_names=transfer.get_feature_names_out)

y_predict: 
 [1 0 0 1 1 0 1 0 0 1 1 1 0 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 1 1 1 0 0
 0 0 0 0 0 1 1 1 0 0 0 1 0 1 0 1 0 0 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0
 0 1 0 0 0 1 0 0 0 1 0 0 1 0 0 0 1 0 0 0 1 0 1 0 1 0 1 1 0 0 1 0 0 1 0 0 0
 0 1 1 0 0 1 0 0 0 0 1 0 0 1 0 0 0 1 1 0 0 0 1 0 1 1 1 0 0 1 0 0 0 0 0 0 0
 0 0 1 0 1 1 0 1 1 0 1 1 0 0 0 0 0 1 0 1 0 0 0 1 0 1 1 1 1 1 0 0 0 1 1 0 1
 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0 1 0 0 0 1 0 0 1 0 0 0 0 1 1 0 1 0 0 0 1
 1]
直接比对真实值和预测值: 
 816    False
789     True
869    False
235    False
473     True
       ...  
174     True
723     True
350     True
399     True
194     True
Name: Survived, Length: 223, dtype: bool
准确率: 
 0.7802690582959642


InvalidParameterError: The 'feature_names' parameter of export_graphviz must be an array-like or None. Got <bound method DictVectorizer.get_feature_names_out of DictVectorizer()> instead.

In [114]:
#随机森林对泰坦尼克号乘客的生存进行预测
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

In [115]:
estimator = RandomForestClassifier()
#加入网格搜索与交叉验证
param_dict = {'n_estimators':[120,200,300,500,800,1200],'max_depth':[5,8,15,25,30]}
estimator = GridSearchCV(estimator,param_grid=param_dict,cv=3)
estimator.fit(x_train,y_train)
#5.模型评估
#方法一：直接比对真实值预测值
y_predict = estimator.predict(x_test)
print('y_predict: \n',y_predict)
print('直接比对真实值和预测值: \n',y_test == y_predict)
#方法二:计算准确率
score = estimator.score(x_test,y_test)
print('准确率: \n',score)
print('最佳参数: \n',estimator.best_params_)
print('最佳结果:\n',estimator.best_score_)
print('最佳估计器: \n',estimator.best_estimator_)
print('交叉验证结果:\n',estimator.cv_results_)

y_predict: 
 [1 0 0 1 1 0 1 0 0 1 1 1 0 0 0 0 1 0 1 0 1 0 1 0 0 0 0 0 1 0 0 0 1 1 1 0 0
 0 0 0 0 1 1 1 1 0 0 0 1 0 1 0 1 0 0 0 0 0 1 1 0 0 1 1 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 1 0 0 1 1 1 0 0 1 0 0 1 0 0 0
 0 1 1 1 0 1 0 0 1 0 1 0 0 1 0 0 0 1 1 0 0 0 1 0 1 1 1 0 0 1 0 0 0 0 0 0 1
 0 0 1 0 1 0 0 1 1 0 1 1 0 0 0 0 0 1 0 1 0 1 0 1 0 0 1 1 1 1 0 0 0 1 1 0 0
 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 1 0 1 1 0 0 1 0 0 1 0 1 0 1 1 1 0 1 0 0 0 1
 1]
直接比对真实值和预测值: 
 816    False
789     True
869    False
235    False
473     True
       ...  
174     True
723     True
350     True
399     True
194     True
Name: Survived, Length: 223, dtype: bool
准确率: 
 0.7668161434977578
最佳参数: 
 {'max_depth': 5, 'n_estimators': 500}
最佳结果:
 0.8204056074011231
最佳估计器: 
 RandomForestClassifier(max_depth=5, n_estimators=500)
交叉验证结果:
 {'mean_fit_time': array([0.12164895, 0.20244678, 0.30705277, 0.51613386, 0.82857831,
       1.22661837, 0.14662751, 0.21591369, 0.35189597, 0.55797299,
       0.881